# ASEAN PTCST-v2 — forecast evaluation

Evaluate the five saved PTCST-v2 seeds with the locked forecast metric protocol. This notebook does not rebuild data or retrain a model.

In [ ]:
# Cell 1 — mount Drive and clone current code
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import subprocess, sys, shutil, pandas as pd
REPO = Path('/content/kltn')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/maiphuowng205/kltn.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-colab.txt')],check=True)
print('Commit:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

In [ ]:
# Cell 2 — restore only the saved forecast runs
DRIVE_RUN = Path('/content/drive/MyDrive/kltn/asean_v2_development/pooled_ptcst')
LOCAL_RUN = Path('/content/asean_v2_forecast_runs')
if not DRIVE_RUN.exists(): raise FileNotFoundError(f'Missing saved V2 run: {DRIVE_RUN}')
if LOCAL_RUN.exists(): shutil.rmtree(LOCAL_RUN)
shutil.copytree(DRIVE_RUN, LOCAL_RUN)
seeds = [7,19,31,43,59]
for seed in seeds:
    path = LOCAL_RUN/f'seed_{seed}'/'development_predictions.npz'
    print(seed, path.exists())
    if not path.exists(): raise FileNotFoundError(path)

In [ ]:
# Cell 3 — locked forecast metrics for all five seeds
METRICS = Path('/content/asean_v2_forecast_metrics')
command = [sys.executable, str(REPO/'scripts/evaluate_asean_v2_forecasts.py'), '--output-dir', str(METRICS)]
for seed in seeds:
    command += ['--input', f'PTCST-v2_seed_{seed}={LOCAL_RUN/f"seed_{seed}"/"development_predictions.npz"}']
subprocess.run(command, check=True)
summary = pd.read_csv(METRICS/'forecast_metrics_summary.csv')
summary

In [ ]:
# Cell 4 — seed stability summary and save it to Drive
primary = ['mean_spearman_ic','median_spearman_ic','ic_hit_rate','icir','top_minus_bottom_5d_bps','dispersion_ratio','calibration_slope']
seed_stability = summary.groupby('country')[primary].agg(['mean','std']).round(6)
display(seed_stability)
DRIVE_OUT = Path('/content/drive/MyDrive/kltn/asean_v2_development/forecast_evaluation')
if DRIVE_OUT.exists(): shutil.rmtree(DRIVE_OUT)
shutil.copytree(METRICS, DRIVE_OUT)
seed_stability.to_csv(DRIVE_OUT/'ptcst_v2_seed_stability.csv')
print('Saved to:', DRIVE_OUT)